In [2]:
from transformers import AutoImageProcessor, AutoModel
import torch
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
processor_rag = AutoImageProcessor.from_pretrained("facebook/dinov2-base")
model_rag = AutoModel.from_pretrained("facebook/dinov2-base")
model_rag.to(device)

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Dinov2Model(
  (embeddings): Dinov2Embeddings(
    (patch_embeddings): Dinov2PatchEmbeddings(
      (projection): Conv2d(3, 768, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Dinov2Encoder(
    (layer): ModuleList(
      (0-11): 12 x Dinov2Layer(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attention): Dinov2Attention(
          (attention): Dinov2SelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
          )
          (output): Dinov2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (layer_scale1): Dinov2LayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06,

In [ ]:
# example 未来可以全部embedding
user_set_style = "yzq"
rag_folder_path_train = f"calli-kaggle/data/data/train/{user_set_style}"
rag_folder_path_test = f"calli-kaggle/data/data/test/{user_set_style}"

print("train 搜索文件夹:", rag_folder_path_train)
print("test 搜索文件夹:", rag_folder_path_test)

train 搜索文件夹: calli-kaggle/data/data/train/yzq
test 搜索文件夹: calli-kaggle/data/data/test/yzq


In [5]:
from pathlib import Path
import json
from PIL import Image
from tqdm import tqdm
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}


def _collect_image_paths(folder_path):
    folder_path = Path(folder_path)
    return sorted(
        p for p in folder_path.rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )


def _embed_images(image_paths, batch_size=16, desc="embedding"):
    model_rag.eval()
    model_device = next(model_rag.parameters()).device
    embeddings = []
    rel_paths = []

    batch_iter = range(0, len(image_paths), batch_size)
    for i in tqdm(batch_iter, total=max(1, len(image_paths) // batch_size + (1 if len(image_paths) % batch_size else 0)), desc=desc):
        batch_paths = image_paths[i:i + batch_size]
        batch_images = [Image.open(p).convert("RGB") for p in batch_paths]
        inputs = processor_rag(images=batch_images, return_tensors="pt")
        inputs = {k: v.to(model_device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model_rag(**inputs)
            batch_emb = outputs.last_hidden_state[:, 0, :].detach().cpu()

        embeddings.append(batch_emb)
        rel_paths.extend([str(p) for p in batch_paths])

    if embeddings:
        embeddings = torch.cat(embeddings, dim=0)
    else:
        embeddings = torch.empty((0, model_rag.config.hidden_size))

    return {
        "image_paths": rel_paths,
        "embeddings": embeddings,
    }

train_image_paths = _collect_image_paths(rag_folder_path_train)
test_image_paths = _collect_image_paths(rag_folder_path_test)

print(f"train 图片数: {len(train_image_paths)}")
print(f"test 图片数: {len(test_image_paths)}")

train_embedding_result = _embed_images(train_image_paths, desc="Embedding train")
test_embedding_result = _embed_images(test_image_paths, desc="Embedding test")

print(f"train embedding 数量: {len(train_embedding_result['image_paths'])}")
print(f"test embedding 数量: {len(test_embedding_result['image_paths'])}")

train 图片数: 5405
test 图片数: 1351


Embedding test: 100%|██████████| 85/85 [00:18<00:00,  4.58it/s]

train embedding 数量: 5405
test embedding 数量: 1351


In [8]:
from pathlib import Path
import torch, json

save_root = Path("embedding")
save_root_train = save_root / "train" / user_set_style
save_root_test = save_root / "test" / user_set_style
save_root_train.mkdir(parents=True, exist_ok=True)
save_root_test.mkdir(parents=True, exist_ok=True)

# 使用 notebook 中的 user_set_style，若不存在则交互输入
name = user_set_style if 'user_set_style' in globals() else input("请输入要保存的大师名称: ")

# 保存 train embedding 和路径
train_emb = train_embedding_result.get("embeddings") if isinstance(train_embedding_result, dict) else None
train_paths = train_embedding_result.get("image_paths") if isinstance(train_embedding_result, dict) else None
if train_emb is not None:
    torch.save(train_emb, save_root_train / f"{name}.pt")
if train_paths is not None:
    with open(save_root_train / f"{name}_paths.json", "w", encoding="utf-8") as f:
        json.dump(train_paths, f, ensure_ascii=False)

# 保存 test embedding 和路径
test_emb = test_embedding_result.get("embeddings") if isinstance(test_embedding_result, dict) else None
test_paths = test_embedding_result.get("image_paths") if isinstance(test_embedding_result, dict) else None
if test_emb is not None:
    torch.save(test_emb, save_root_test / f"{name}.pt")
if test_paths is not None:
    with open(save_root_test / f"{name}_paths.json", "w", encoding="utf-8") as f:
        json.dump(test_paths, f, ensure_ascii=False)

print("Saved embeddings to", save_root_train, save_root_test)
# 在embedding文件夹中创建yzq.txt,内容空，用于标记已完成embedding处理
yzq_txt_path = save_root / "yzq.txt"
yzq_txt_path.touch(exist_ok=True)
print(f"Created marker file: {yzq_txt_path}")

Saved embeddings to embedding/train/yzq embedding/test/yzq
Created marker file: embedding/yzq.txt
